# EXP03: Trail 2D 网格扫描 ($0.50 + $0.30)

**合并**: NB06 + NB07

TrailAct x TrailDist = 6 x 5 = 30 变体，两种点差各一轮 = 60 变体

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
from backtest.stats import print_ranked
import pandas as pd

data = DataBundle.load_default()

BASE_KWARGS = {
    "sl_atr_mult": 4.5,
    "tp_atr_mult": 5.0,
    "keltner_adx_threshold": 18,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
}
print('Data loaded')

## Part 1: $0.50 spread

In [ ]:
results_050 = []
for t_act in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    for t_dist in [0.15, 0.20, 0.25, 0.30, 0.35]:
        kw = {**BASE_KWARGS, "trailing_activate_atr": t_act, "trailing_distance_atr": t_dist, "spread_cost": 0.50}
        r = run_variant(data, f"T{t_act}/D{t_dist}_$0.50", **kw)
        r['trail_act'] = t_act
        r['trail_dist'] = t_dist
        r['spread'] = 0.50
        results_050.append(r)

print_ranked(results_050)

In [ ]:
df = pd.DataFrame([{'trail_act': r['trail_act'], 'trail_dist': r['trail_dist'],
                     'sharpe': r['sharpe'], 'total_pnl': r['total_pnl']} for r in results_050])
pivot = df.pivot_table(index='trail_act', columns='trail_dist', values='sharpe')
print("Sharpe Heatmap ($0.50 spread):")
print(pivot.round(2))
pivot_pnl = df.pivot_table(index='trail_act', columns='trail_dist', values='total_pnl')
print("\nPnL Heatmap ($0.50):")
print(pivot_pnl.round(0))

## Part 2: $0.30 spread

In [ ]:
results_030 = []
for t_act in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    for t_dist in [0.15, 0.20, 0.25, 0.30, 0.35]:
        kw = {**BASE_KWARGS, "trailing_activate_atr": t_act, "trailing_distance_atr": t_dist, "spread_cost": 0.30}
        r = run_variant(data, f"T{t_act}/D{t_dist}_$0.30", **kw)
        r['trail_act'] = t_act
        r['trail_dist'] = t_dist
        r['spread'] = 0.30
        results_030.append(r)

print_ranked(results_030)

In [ ]:
df2 = pd.DataFrame([{'trail_act': r['trail_act'], 'trail_dist': r['trail_dist'],
                      'sharpe': r['sharpe']} for r in results_030])
pivot2 = df2.pivot_table(index='trail_act', columns='trail_dist', values='sharpe')
print("Sharpe Heatmap ($0.30 spread):")
print(pivot2.round(2))

## Part 3: 两种点差最优参数对比

In [ ]:
best_050 = sorted(results_050, key=lambda r: r['sharpe'], reverse=True)[0]
best_030 = sorted(results_030, key=lambda r: r['sharpe'], reverse=True)[0]
print(f"Best $0.50: TrailAct={best_050['trail_act']}, TrailDist={best_050['trail_dist']}, Sharpe={best_050['sharpe']:.2f}")
print(f"Best $0.30: TrailAct={best_030['trail_act']}, TrailDist={best_030['trail_dist']}, Sharpe={best_030['sharpe']:.2f}")
print(f"Same optimal? {best_050['trail_act']==best_030['trail_act'] and best_050['trail_dist']==best_030['trail_dist']}")

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp03_results.json', 'w') as f:
    json.dump(sanitize_for_json(results_050 + results_030), f, indent=2)
print('Saved')